# ETL and Data Pipelines: Solutions

Instructor copy. Run the setup cell first, then the sections in order. The mini
project at the end is a complete, runnable pipeline.

In [ ]:
# SETUP: rebuild the three sources
import os, sqlite3, json, threading, time, glob, shutil
import pandas as pd
import numpy as np
from contextlib import closing
from datetime import date, datetime
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.parse import urlparse, parse_qs

for folder in ["landing", "raw", "warehouse_files"]:
    if os.path.isdir(folder):
        shutil.rmtree(folder)
os.makedirs("landing", exist_ok=True)

DAILY = {
    "2026-05-01": ["5001,101,Laptop Pro,1,450000,NG,completed",
                   "5002,102,Phone X,2,180000,GH,completed",
                   "5003,101,Wireless Mouse,3,8000,NG,completed"],
    "2026-05-02": ["5004,103,Laptop Pro,1,520000,NG,pending",
                   "5005,104,Monitor 27,2,95000,KE,completed",
                   "5006,102,Wireless Mouse,1,N/A,GH,cancelled"],
    "2026-05-03": ["5007,103,Keyboard,2,15000,NG,completed",
                   "5008,,Phone X,1,180000,KE,completed",
                   "5009,999,Headset,4,22000,GH,completed",
                   "5007,103,Keyboard,2,15000,NG,completed"],
}
for day, rows in DAILY.items():
    with open(f"landing/orders_{day}.csv", "w", encoding="utf-8") as f:
        f.write("order_id,customer_id,item,quantity,unit_price,country,status\n")
        f.write("\n".join(rows) + "\n")

if os.path.exists("source.db"):
    os.remove("source.db")
with closing(sqlite3.connect("source.db")) as source:
    source.executescript('''CREATE TABLE customers (
        customer_id INTEGER PRIMARY KEY, name TEXT, segment TEXT,
        country TEXT, updated_at TEXT);''')
    source.executemany("INSERT INTO customers VALUES (?, ?, ?, ?, ?)", [
        (101, "Ahmed Oladapo", "enterprise", "NG", "2026-04-01T09:00:00"),
        (102, "Lahla Mensah", "smb", "GH", "2026-04-02T09:00:00"),
        (103, "Judith Eze", "enterprise", "NG", "2026-04-03T09:00:00"),
        (104, "Uche Okafor", "smb", "KE", "2026-05-01T09:00:00"),
        (105, "Grace Wanjiru", "smb", "KE", "2026-05-02T09:00:00")])
    source.commit()

PRODUCTS = [
    {"sku": f"SKU-{i:03d}", "name": n, "category": c, "cost": cost,
     "supplier": {"name": s, "country": co}}
    for i, (n, c, cost, s, co) in enumerate([
        ("Laptop Pro", "Electronics", 380000, "TechDist", "NG"),
        ("Phone X", "Electronics", 140000, "TechDist", "NG"),
        ("Wireless Mouse", "Accessories", 5200, "GadgetHub", "GH"),
        ("Monitor 27", "Electronics", 71000, "TechDist", "NG"),
        ("Keyboard", "Accessories", 9800, "GadgetHub", "GH"),
        ("Headset", "Accessories", 15400, "SoundCo", "KE"),
        ("Webcam", "Accessories", 24000, "SoundCo", "KE")], start=1)
]
PAGE_SIZE = 3


class Handler(BaseHTTPRequestHandler):
    def log_message(self, *args):
        pass

    def do_GET(self):
        parsed = urlparse(self.path)
        if parsed.path != "/api/products":
            body, code = json.dumps({"error": "not found"}).encode(), 404
        else:
            page = int(parse_qs(parsed.query).get("page", ["1"])[0])
            pages = max(1, -(-len(PRODUCTS) // PAGE_SIZE))
            start = (page - 1) * PAGE_SIZE
            body = json.dumps({
                "info": {"count": len(PRODUCTS), "pages": pages,
                         "next": f"{API_URL}/api/products?page={page+1}"
                                 if page < pages else None},
                "results": PRODUCTS[start:start + PAGE_SIZE]}).encode()
            code = 200
        self.send_response(code)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)


class QuietServer(ThreadingHTTPServer):
    daemon_threads = True

    def handle_error(self, request, client_address):
        pass


_api = QuietServer(("127.0.0.1", 0), Handler)
threading.Thread(target=_api.serve_forever, daemon=True).start()
API_URL = f"http://127.0.0.1:{_api.server_address[1]}"
time.sleep(0.3)

pd.DataFrame({"column_name": ["10.5", "20.0", "30.5", None, "50.0"],
              "column1": [1, 2, 3, 4, 5],
              "column2": [10, 20, 30, 40, 50]}).to_csv("example.csv", index=False)
df = pd.read_csv("example.csv")

print(f"setup done. api at {API_URL}")

## Extract: tasks

In [ ]:
# Task 1
def extract_files(pattern="landing/orders_*.csv"):
    """Read every matching CSV as text, tagging each row with its source file."""
    frames = []
    for path in sorted(glob.glob(pattern)):
        frame = pd.read_csv(path, dtype=str, encoding="utf-8")
        frame["_source_file"] = os.path.basename(path)
        frames.append(frame)
        print(f"  {os.path.basename(path)}: {len(frame)} row(s)")
    if not frames:
        raise FileNotFoundError(f"no files matched {pattern!r}")
    return pd.concat(frames, ignore_index=True)

raw_orders = extract_files()
print(f"total: {len(raw_orders)}")

In [ ]:
# Task 2
def extract_customers(since=None, db="source.db"):
    """Read customers, incrementally when a watermark is given."""
    with closing(sqlite3.connect(db)) as connection:
        if since:
            return pd.read_sql(
                "SELECT * FROM customers WHERE updated_at > ? ORDER BY updated_at",
                connection, params=(since,))
        return pd.read_sql("SELECT * FROM customers", connection)

full = extract_customers()
recent = extract_customers(since="2026-04-30T00:00:00")
print(f"full: {len(full)}, since 2026-04-30: {len(recent)}")

In [ ]:
# Task 3
import requests

def extract_products(url, max_pages=20):
    """Page through the products API."""
    records, pages = [], 0
    while url and pages < max_pages:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        payload = response.json()
        records.extend(payload["results"])
        pages += 1
        url = payload["info"]["next"]
    print(f"{len(records)} product(s) over {pages} page(s)")
    return records

products_raw = extract_products(f"{API_URL}/api/products")

In [ ]:
# DE Task 4: land the raw extracts
RUN_DATE = date.today().isoformat()
raw_dir = os.path.join("raw", f"date={RUN_DATE}")
os.makedirs(raw_dir, exist_ok=True)

raw_orders.to_csv(os.path.join(raw_dir, "orders.csv"), index=False)
full.to_csv(os.path.join(raw_dir, "customers.csv"), index=False)
with open(os.path.join(raw_dir, "products.json"), "w", encoding="utf-8") as f:
    json.dump(products_raw, f, indent=2)

for path in sorted(glob.glob("raw/**/*", recursive=True)):
    if os.path.isfile(path):
        print(f"{path}  ({os.path.getsize(path)} bytes)")

In [ ]:
# DE Task 5: the manifest
def build_manifest(sources, run_date=RUN_DATE):
    """Record row counts, columns and null counts per source."""
    return {"run_date": run_date, "sources": {
        name: {"rows": len(frame),
               "columns": list(frame.columns),
               "null_counts": {c: int(frame[c].isna().sum()) for c in frame.columns}}
        for name, frame in sources.items()}}

manifest = build_manifest({"orders": raw_orders, "customers": full,
                           "products": pd.json_normalize(products_raw)})
with open(os.path.join(raw_dir, "_manifest.json"), "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

for name, info in manifest["sources"].items():
    print(f"{name}: {info['rows']} rows")

In [ ]:
# DE Task 6: volume check against a previous run
def check_volume(manifest, previous, tolerance=0.5):
    """Warn when a source's row count moves by more than the tolerance."""
    for name, info in manifest["sources"].items():
        before = previous.get("sources", {}).get(name, {}).get("rows")
        if before is None:
            print(f"{name}: no previous run to compare against")
            continue
        if before == 0:
            continue
        change = (info["rows"] - before) / before
        flag = "WARNING" if abs(change) > tolerance else "ok"
        print(f"{flag:<8} {name}: {before} -> {info['rows']} ({change:+.0%})")

fake_previous = {"sources": {"orders": {"rows": 10}, "customers": {"rows": 5},
                             "products": {"rows": 40}}}
check_volume(manifest, fake_previous)

## Transform: tasks

In [ ]:
# Task 1
def standardise(df):
    """Normalise column names and strip whitespace from text columns."""
    out = df.copy()
    out.columns = out.columns.str.strip().str.lower().str.replace(" ", "_", regex=False)
    for column in out.columns:
        if not pd.api.types.is_numeric_dtype(out[column]):
            out[column] = (out[column].astype(str).str.strip()
                           .replace({"nan": None, "None": None, "": None}))
    return out

standardised = standardise(raw_orders)
print(list(standardised.columns))

In [ ]:
# Task 2
def coerce_types(df):
    """Convert numeric columns, reporting how many values became null."""
    out = df.copy()
    report = {}
    for column in ["order_id", "customer_id", "quantity", "unit_price"]:
        before = out[column].isna().sum()
        out[column] = pd.to_numeric(out[column], errors="coerce")
        report[column] = int(out[column].isna().sum() - before)
    for column, n in report.items():
        if n:
            print(f"  {column}: {n} value(s) coerced to null")
    return out, report

typed, coercion_report = coerce_types(standardised)
print(typed.dtypes.to_dict())

In [ ]:
# Task 3
def deduplicate(df, key="order_id"):
    """Drop repeats of the business key, returning (kept, dropped)."""
    dropped = df[df.duplicated(subset=[key], keep="first")].copy()
    kept = df.drop_duplicates(subset=[key], keep="first")
    if len(dropped):
        print(f"  dropped {len(dropped)} duplicate {key}(s): "
              f"{sorted(dropped[key].dropna().astype(int).tolist())}")
    return kept, dropped

deduped, dupes = deduplicate(typed)
print(f"{len(typed)} -> {len(deduped)}")

In [ ]:
# DE Task 4
VALID_STATUSES = {"completed", "pending", "cancelled"}

def validate(df, known_customers):
    """Split into (clean, rejected). Rejected rows carry a reason."""
    reasons = []
    for _, row in df.iterrows():
        if pd.isna(row["order_id"]):
            reasons.append("missing order_id")
        elif pd.isna(row["customer_id"]):
            reasons.append("missing customer_id")
        elif int(row["customer_id"]) not in known_customers:
            reasons.append(f"unknown customer {int(row['customer_id'])}")
        elif pd.isna(row["unit_price"]):
            reasons.append("unit_price could not be parsed")
        elif pd.isna(row["quantity"]) or row["quantity"] < 1:
            reasons.append(f"bad quantity {row['quantity']}")
        elif row["status"] not in VALID_STATUSES:
            reasons.append(f"unexpected status {row['status']!r}")
        else:
            reasons.append(None)

    tagged = df.assign(_reason=reasons)
    clean = tagged[tagged["_reason"].isna()].drop(columns=["_reason"]).copy()
    rejected = tagged[tagged["_reason"].notna()].copy()
    return clean, rejected

known = set(full["customer_id"])
clean, rejected = validate(deduped, known)
print(f"clean {len(clean)}, rejected {len(rejected)}")
for _, row in rejected.iterrows():
    label = int(row["order_id"]) if pd.notna(row["order_id"]) else "?"
    print(f"  order {label}: {row['_reason']}")

In [ ]:
# DE Task 5
def enrich(orders, customers, products=None):
    """Join the dimensions and add the derived columns."""
    out = orders.merge(customers[["customer_id", "name", "segment"]],
                       on="customer_id", how="left", validate="many_to_one")

    if products is not None:
        out = out.merge(products[["name", "cost"]].rename(
            columns={"name": "item", "cost": "unit_cost"}),
            on="item", how="left", validate="many_to_one")
    else:
        out["unit_cost"] = np.nan

    return out.assign(
        revenue=lambda d: d["quantity"] * d["unit_price"],
        vat=lambda d: d["revenue"] * 0.075,
        total=lambda d: d["revenue"] + d["vat"],
        margin=lambda d: d["revenue"] - (d["quantity"] * d["unit_cost"]),
        order_date=lambda d: d["_source_file"].str.extract(r"(\d{4}-\d{2}-\d{2})")[0],
        value_band=lambda d: np.select(
            [d["revenue"] > 400000, d["revenue"] > 50000],
            ["large", "medium"], default="small"),
    )

products_df = pd.json_normalize(products_raw, sep="_")
enriched = enrich(clean, full, products_df)
enriched[["order_id", "name", "item", "revenue", "margin", "order_date", "value_band"]]

In [ ]:
# DE Task 6
def quality_gates(clean, rejected, coercion_report, max_reject_rate=0.40):
    """Return (passed, messages). A failed gate must stop the load."""
    messages, passed = [], True
    total = len(clean) + len(rejected)

    rate = len(rejected) / total if total else 0
    if rate > max_reject_rate:
        passed = False
        messages.append(f"GATE FAILED: {rate:.0%} rejected, limit {max_reject_rate:.0%}")
    else:
        messages.append(f"ok: rejection rate {rate:.0%}")

    if clean.empty:
        passed = False
        messages.append("GATE FAILED: no clean rows")
    if clean["order_id"].duplicated().any():
        passed = False
        messages.append("GATE FAILED: duplicate order_id survived")
    if "revenue" in clean and (clean["revenue"] < 0).any():
        passed = False
        messages.append("GATE FAILED: negative revenue")
    if "order_date" in clean and clean["order_date"].isna().any():
        passed = False
        messages.append("GATE FAILED: missing order_date")

    for column, n in coercion_report.items():
        if n:
            messages.append(f"warning: {n} coerced in {column}")
    return passed, messages

passed, messages = quality_gates(enriched, rejected, coercion_report)
for m in messages:
    print(m)
print("PROCEED" if passed else "STOP")

print()
print("now with a threshold that fails:")
strict_passed, strict_messages = quality_gates(
    enriched, rejected, coercion_report, max_reject_rate=0.10)
for m in strict_messages:
    print(m)
print("PROCEED" if strict_passed else "STOP, do not load")

## Load: tasks

In [ ]:
# Tasks 1 to 4: the warehouse, the upsert, verification and the rejects table
if os.path.exists("warehouse.db"):
    os.remove("warehouse.db")

with closing(sqlite3.connect("warehouse.db")) as wh:
    wh.executescript('''
    CREATE TABLE fact_orders (
        order_id INTEGER PRIMARY KEY, customer_id INTEGER, customer_name TEXT,
        segment TEXT, item TEXT, quantity INTEGER, unit_price REAL, revenue REAL,
        vat REAL, total REAL, margin REAL, country TEXT, status TEXT,
        order_date TEXT, value_band TEXT, source_file TEXT, loaded_at TEXT);
    CREATE TABLE dim_customers (
        customer_id INTEGER PRIMARY KEY, name TEXT, segment TEXT, country TEXT,
        updated_at TEXT, loaded_at TEXT);
    CREATE TABLE dim_products (
        sku TEXT PRIMARY KEY, name TEXT, category TEXT, cost REAL,
        supplier TEXT, loaded_at TEXT);
    CREATE TABLE rejected_orders (
        order_id TEXT, reason TEXT, source_file TEXT, rejected_at TEXT);
    CREATE TABLE etl_runs (
        run_id INTEGER PRIMARY KEY AUTOINCREMENT, pipeline TEXT, run_date TEXT,
        started_at TEXT, finished_at TEXT, rows_read INTEGER, rows_loaded INTEGER,
        rows_rejected INTEGER, status TEXT, message TEXT);
    CREATE TABLE etl_watermark (
        source TEXT PRIMARY KEY, last_value TEXT, updated_at TEXT);
    ''')
    wh.commit()

print("warehouse.db created with 6 tables")

In [ ]:
FACT_COLUMNS = ["order_id", "customer_id", "customer_name", "segment", "item",
                "quantity", "unit_price", "revenue", "vat", "total", "margin",
                "country", "status", "order_date", "value_band", "source_file",
                "loaded_at"]

UPSERT_FACT = f'''
INSERT INTO fact_orders ({", ".join(FACT_COLUMNS)})
VALUES ({", ".join("?" for _ in FACT_COLUMNS)})
ON CONFLICT(order_id) DO UPDATE SET
{", ".join(f"{c} = excluded.{c}" for c in FACT_COLUMNS if c != "order_id")}
'''


def _tuples(df, columns):
    """DataFrame to tuples, turning NaN into a real NULL."""
    return [tuple(None if pd.isna(v) else v for v in row)
            for row in df[columns].itertuples(index=False, name=None)]


def load_fact(df, db="warehouse.db"):
    """Upsert into fact_orders. Idempotent."""
    out = df.copy()
    out["customer_name"] = out["name"]
    out["source_file"] = out["_source_file"]
    out["loaded_at"] = datetime.now().isoformat(timespec="seconds")
    with closing(sqlite3.connect(db)) as wh:
        with wh:
            wh.executemany(UPSERT_FACT, _tuples(out, FACT_COLUMNS))
        return wh.execute("SELECT COUNT(*) FROM fact_orders").fetchone()[0]


for attempt in [1, 2, 3]:
    print(f"run {attempt}: {len(enriched)} rows in, {load_fact(enriched)} in warehouse")

In [ ]:
def load_dim_customers(df, db="warehouse.db"):
    """Upsert the customer dimension."""
    out = df.copy()
    out["loaded_at"] = datetime.now().isoformat(timespec="seconds")
    cols = ["customer_id", "name", "segment", "country", "updated_at", "loaded_at"]
    sql = f'''INSERT INTO dim_customers ({", ".join(cols)})
              VALUES ({", ".join("?" for _ in cols)})
              ON CONFLICT(customer_id) DO UPDATE SET
              {", ".join(f"{c} = excluded.{c}" for c in cols if c != "customer_id")}'''
    with closing(sqlite3.connect(db)) as wh:
        with wh:
            wh.executemany(sql, _tuples(out, cols))
        return wh.execute("SELECT COUNT(*) FROM dim_customers").fetchone()[0]


def load_dim_products(records, db="warehouse.db"):
    """Upsert the product dimension from the raw API records."""
    now = datetime.now().isoformat(timespec="seconds")
    payload = [(r["sku"], r["name"], r["category"], r["cost"],
                r["supplier"]["name"], now) for r in records]
    sql = '''INSERT INTO dim_products (sku, name, category, cost, supplier, loaded_at)
             VALUES (?, ?, ?, ?, ?, ?)
             ON CONFLICT(sku) DO UPDATE SET
             name = excluded.name, category = excluded.category, cost = excluded.cost,
             supplier = excluded.supplier, loaded_at = excluded.loaded_at'''
    with closing(sqlite3.connect(db)) as wh:
        with wh:
            wh.executemany(sql, payload)
        return wh.execute("SELECT COUNT(*) FROM dim_products").fetchone()[0]


def load_rejects(rejected, db="warehouse.db"):
    """Replace the rejects table with this run's rejections."""
    now = datetime.now().isoformat(timespec="seconds")
    payload = [(str(r["order_id"]), r["_reason"], r["_source_file"], now)
               for _, r in rejected.iterrows()]
    with closing(sqlite3.connect(db)) as wh:
        with wh:
            wh.execute("DELETE FROM rejected_orders")
            if payload:
                wh.executemany("INSERT INTO rejected_orders VALUES (?, ?, ?, ?)", payload)
        return wh.execute("SELECT COUNT(*) FROM rejected_orders").fetchone()[0]


def verify_load(df, db="warehouse.db"):
    """Compare expected against actual row count and revenue."""
    with closing(sqlite3.connect(db)) as wh:
        rows, revenue = wh.execute(
            "SELECT COUNT(*), COALESCE(SUM(revenue), 0) FROM fact_orders").fetchone()
    ok = rows == len(df) and abs(revenue - float(df["revenue"].sum())) < 0.01
    print(f"rows {len(df)} vs {rows}, revenue {df['revenue'].sum():,.2f} vs {revenue:,.2f}")
    print("VERIFIED" if ok else "MISMATCH")
    return ok


print(f"customers: {load_dim_customers(full)}")
print(f"products:  {load_dim_products(products_raw)}")
print(f"rejects:   {load_rejects(rejected)}")
verify_load(enriched)

In [ ]:
# DE Task 5: change a status upstream, re-run, confirm an UPDATE not an INSERT
before = load_fact(enriched)

path = "landing/orders_2026-05-01.csv"
text = open(path, encoding="utf-8").read().replace(
    "5001,101,Laptop Pro,1,450000,NG,completed",
    "5001,101,Laptop Pro,1,450000,NG,cancelled")
open(path, "w", encoding="utf-8").write(text)

raw2 = extract_files()
typed2, report2 = coerce_types(standardise(raw2))
deduped2, _ = deduplicate(typed2)
clean2, rejected2 = validate(deduped2, known)
enriched2 = enrich(clean2, full, products_df)
after = load_fact(enriched2)

with closing(sqlite3.connect("warehouse.db")) as wh:
    status = wh.execute(
        "SELECT status FROM fact_orders WHERE order_id = 5001").fetchone()[0]

print(f"rows before {before}, after {after}")
print(f"order 5001 status is now {status!r}")
print("updated in place" if before == after else "DUPLICATED, investigate")

In [ ]:
# DE Task 6: partitioned files, reconciled against the database
def load_files(df, base="warehouse_files"):
    """Write one CSV per order_date partition."""
    written = []
    for partition, group in df.dropna(subset=["order_date"]).groupby("order_date"):
        folder = os.path.join(base, f"date={partition}")
        os.makedirs(folder, exist_ok=True)
        path = os.path.join(folder, "orders.csv")
        group.to_csv(path, index=False)
        written.append((path, len(group)))
    return written

for path, rows in load_files(enriched2):
    print(f"  {path} ({rows} rows)")

back = pd.concat([pd.read_csv(p) for p in
                  sorted(glob.glob("warehouse_files/**/*.csv", recursive=True))],
                 ignore_index=True)

with closing(sqlite3.connect("warehouse.db")) as wh:
    db_revenue = wh.execute("SELECT SUM(revenue) FROM fact_orders").fetchone()[0]

print(f"\nfiles: {len(back)} rows, {back['revenue'].sum():,.2f}")
print(f"db:    revenue {db_revenue:,.2f}")
print("reconciled" if abs(back["revenue"].sum() - db_revenue) < 0.01 else "MISMATCH")

## Pipeline mechanics: tasks

In [ ]:
# Task 1: logging
import logging, sys

logger = logging.getLogger("dec_pipeline")
logger.setLevel(logging.INFO)
logger.handlers.clear()

formatter = logging.Formatter(
    "%(asctime)s | %(levelname)-8s | %(message)s", datefmt="%Y-%m-%d %H:%M:%S")

console = logging.StreamHandler(sys.stdout)
console.setFormatter(formatter)
logger.addHandler(console)

file_handler = logging.FileHandler("pipeline.log", encoding="utf-8")
file_handler.setFormatter(formatter)
logger.addHandler(file_handler)

logger.info("extract complete: 10 rows")
logger.warning("1 value coerced to null in unit_price")
logger.error("quality gate failed")

In [ ]:
# Task 2: config
CONFIG = {
    "pipeline": "orders_daily",
    "sources": {"files": {"pattern": "landing/orders_*.csv"},
                "database": {"path": "source.db", "table": "customers"},
                "api": {"url": f"{API_URL}/api/products"}},
    "target": {"path": "warehouse.db"},
    "quality": {"max_reject_rate": 0.40},
}
with open("pipeline_config.json", "w", encoding="utf-8") as f:
    json.dump(CONFIG, f, indent=2)

with open("pipeline_config.json", encoding="utf-8") as f:
    loaded = json.load(f)
print(loaded["quality"])

In [ ]:
# Task 3: retry decorator
import functools

def with_retry(attempts=3, base_delay=0.1, exceptions=(Exception,)):
    """Retry the wrapped function with exponential backoff."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            delay = base_delay
            for attempt in range(1, attempts + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions:
                    if attempt == attempts:
                        raise
                    logger.warning(f"{func.__name__} attempt {attempt} failed, "
                                   f"retrying in {delay:.1f}s")
                    time.sleep(delay)
                    delay *= 2
        return wrapper
    return decorator

_calls = {"n": 0}

@with_retry(attempts=4)
def flaky():
    _calls["n"] += 1
    if _calls["n"] < 3:
        raise ConnectionError("source unreachable")
    return f"ok on attempt {_calls['n']}"

print(flaky())

In [ ]:
# DE Task 4: the run log
def log_run(pipeline, run_date, started, finished, read, loaded, rejected_count,
            status, message, db="warehouse.db"):
    """Record one run in etl_runs."""
    with closing(sqlite3.connect(db)) as wh:
        with wh:
            wh.execute('''INSERT INTO etl_runs
                (pipeline, run_date, started_at, finished_at, rows_read, rows_loaded,
                 rows_rejected, status, message) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)''',
                (pipeline, run_date, started, finished, read, loaded,
                 rejected_count, status, message))


def show_runs(db="warehouse.db"):
    with closing(sqlite3.connect(db)) as wh:
        rows = wh.execute("SELECT run_id, run_date, status, rows_read, rows_loaded, "
                          "rows_rejected, message FROM etl_runs ORDER BY run_id").fetchall()
    print(f"{'id':<4}{'date':<13}{'status':<10}{'read':<6}{'load':<6}{'rej':<5}message")
    print("-" * 78)
    for r in rows:
        print(f"{r[0]:<4}{r[1]:<13}{r[2]:<10}{r[3]:<6}{r[4]:<6}{r[5]:<5}{r[6]}")


now = datetime.now().isoformat(timespec="seconds")
log_run("orders_daily", "2026-05-01", now, now, 10, 8, 2, "success", "ok")
log_run("orders_daily", "2026-05-02", now, now, 10, 0, 6, "failed", "quality gate failed")
show_runs()

In [ ]:
# DE Task 5: backfill
def run_for_date(run_date):
    """Process one day's file. Returns the row count, or 0 if there is no file."""
    path = f"landing/orders_{run_date}.csv"
    if not os.path.exists(path):
        logger.warning(f"no file for {run_date}, skipping")
        return 0
    rows = len(pd.read_csv(path))
    logger.info(f"{run_date}: {rows} row(s)")
    return rows

total = sum(run_for_date(d) for d in
            ["2026-05-01", "2026-05-02", "2026-05-03", "2026-05-04"])
print(f"backfilled {total} row(s)")

## Mini project: the complete pipeline

In [ ]:
def run(run_date=None, dry_run=False, max_reject_rate=0.40, db="warehouse.db"):
    """Run the whole pipeline. Idempotent, logged, and safe to re-run."""
    run_date = run_date or date.today().isoformat()
    started = datetime.now().isoformat(timespec="seconds")
    read = loaded = rejected_count = 0

    try:
        logger.info(f"=== {CONFIG['pipeline']} for {run_date}"
                    f"{' (dry run)' if dry_run else ''} ===")

        # EXTRACT
        orders_raw = extract_files(CONFIG["sources"]["files"]["pattern"])
        customers = extract_customers()
        products = extract_products(CONFIG["sources"]["api"]["url"])
        read = len(orders_raw)
        logger.info(f"extracted {read} order(s), {len(customers)} customer(s), "
                    f"{len(products)} product(s)")

        # TRANSFORM
        typed, report = coerce_types(standardise(orders_raw))
        deduped, dropped = deduplicate(typed)
        clean, rejected = validate(deduped, set(customers["customer_id"]))
        products_frame = pd.json_normalize(products, sep="_")
        enriched = enrich(clean, customers, products_frame)
        rejected_count = len(rejected)
        logger.info(f"transformed: {len(enriched)} clean, {rejected_count} rejected, "
                    f"{len(dropped)} duplicate(s) dropped")

        # QUALITY
        passed, messages = quality_gates(enriched, rejected, report, max_reject_rate)
        for message in messages:
            (logger.warning if "FAILED" in message or "warning" in message
             else logger.info)(message)
        if not passed:
            raise ValueError("quality gate failed, nothing was loaded")

        if dry_run:
            logger.info("dry run, nothing written")
            finished = datetime.now().isoformat(timespec="seconds")
            log_run(CONFIG["pipeline"], run_date, started, finished, read, 0,
                    rejected_count, "dry_run", "validated only", db)
            return {"status": "dry_run", "read": read, "would_load": len(enriched)}

        # LOAD
        load_dim_customers(customers, db)
        load_dim_products(products, db)
        total = load_fact(enriched, db)
        load_rejects(rejected, db)
        loaded = len(enriched)
        verify_load(enriched, db)

        finished = datetime.now().isoformat(timespec="seconds")
        log_run(CONFIG["pipeline"], run_date, started, finished, read, loaded,
                rejected_count, "success", f"{total} rows in fact_orders", db)
        logger.info(f"loaded {loaded} row(s), {total} in the warehouse")

        return {"status": "success", "read": read, "loaded": loaded,
                "rejected": rejected_count, "total": total}

    except Exception as e:
        finished = datetime.now().isoformat(timespec="seconds")
        log_run(CONFIG["pipeline"], run_date, started, finished, read, loaded,
                rejected_count, "failed", f"{type(e).__name__}: {e}", db)
        logger.error(f"run failed: {type(e).__name__}: {e}")
        raise


summary = run("2026-05-03")
print()
print(summary)

In [ ]:
# Prove it: three runs, same row count
results = [run("2026-05-03") for _ in range(2)]
print()
for i, r in enumerate(results, start=2):
    print(f"run {i}: loaded {r['loaded']}, total {r['total']}")
print("idempotent" if results[0]["total"] == results[-1]["total"] else "NOT idempotent")

In [ ]:
# Prove the gate stops the load
try:
    run("2026-05-03", max_reject_rate=0.10)
except ValueError as e:
    print(f"\nstopped as designed: {e}")

print()
print("a dry run:")
print(run("2026-05-03", dry_run=True))

In [ ]:
# The run log
show_runs()

In [ ]:
# Clean up
import os, shutil

_api.shutdown()
for name in ["source.db", "warehouse.db", "example.csv", "pipeline.log",
             "pipeline_config.json", "final_output.csv", "transformed_data.csv"]:
    if os.path.isfile(name):
        os.remove(name)
for folder in ["landing", "raw", "warehouse_files"]:
    if os.path.isdir(folder):
        shutil.rmtree(folder)
print("cleaned up")